# KIU Internship Support Services Analysis

This notebook loads the Google Forms CSV export, cleans the data, calculates descriptive statistics, compares readiness scores by support usage, and creates basic charts.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'data' / 'raw' / 'google-forms-export-placeholder.csv'
OUTPUT_DIR = BASE_DIR / 'analysis' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
service_columns = [
    'info_availability', 'communication_quality', 'application_support',
    'cv_support', 'interview_support', 'industry_connections',
    'accessibility', 'overall_satisfaction'
]
readiness_columns = [
    'finding_confidence', 'cv_confidence', 'interview_confidence',
    'employer_expectations', 'readiness_improvement'
]
numeric_columns = ['awareness_score'] + service_columns + readiness_columns

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

for column in ['searched_for_internship', 'completed_internship', 'used_support']:
    df[column] = df[column].astype(str).str.strip().str.title()

df['satisfaction_score'] = df[service_columns].mean(axis=1)
df['readiness_score'] = df[readiness_columns].mean(axis=1)
df[['respondent_id', 'used_support', 'satisfaction_score', 'readiness_score']].head()

In [ ]:
categorical_columns = ['year_of_study', 'school', 'searched_for_internship', 'completed_internship', 'used_support']
frequency_tables = []
for column in categorical_columns:
    table = df[column].value_counts(dropna=False).rename_axis('category').reset_index(name='count')
    table['variable'] = column
    table['percentage'] = (table['count'] / len(df) * 100).round(1)
    frequency_tables.append(table[['variable', 'category', 'count', 'percentage']])

frequencies = pd.concat(frequency_tables, ignore_index=True)
frequencies

In [ ]:
scale_means = df[['awareness_score'] + service_columns + readiness_columns + ['satisfaction_score', 'readiness_score']].mean().round(2)
scale_means

In [ ]:
readiness_by_usage = df.groupby('used_support')['readiness_score'].agg(['count', 'mean', 'std']).round(2)
readiness_by_usage

In [ ]:
used = df.loc[df['used_support'] == 'Yes', 'readiness_score'].dropna()
not_used = df.loc[df['used_support'] == 'No', 'readiness_score'].dropna()

if len(used) >= 2 and len(not_used) >= 2:
    ttest = stats.ttest_ind(used, not_used, equal_var=False)
    print(f"Used support mean: {used.mean():.2f}")
    print(f"Did not use support mean: {not_used.mean():.2f}")
    print(f"t-statistic: {ttest.statistic:.4f}")
    print(f"p-value: {ttest.pvalue:.4f}")
else:
    print('Not enough valid responses in both groups to run a t-test.')

In [ ]:
df['used_support'].value_counts().sort_index().plot(kind='bar')
plt.title('Use of KIU Internship Support Services')
plt.xlabel('Used support services')
plt.ylabel('Number of respondents')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df[service_columns].mean().sort_values().plot(kind='barh')
plt.title('Mean Service Evaluation Scores')
plt.xlabel('Mean score (1-5)')
plt.xlim(1, 5)
plt.tight_layout()
plt.show()

In [ ]:
df.to_csv(OUTPUT_DIR / 'cleaned-survey-data.csv', index=False)
frequencies.to_csv(OUTPUT_DIR / 'categorical-frequencies.csv', index=False)
scale_means.reset_index().rename(columns={'index': 'variable', 0: 'mean'}).to_csv(OUTPUT_DIR / 'scale-means.csv', index=False)
readiness_by_usage.to_csv(OUTPUT_DIR / 'readiness-by-support-usage.csv')